<a href="https://colab.research.google.com/github/tbrechbuehl/Expert-RAG-System/blob/main/expert_rag_irf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
import os

# Execute this code only if in colab
if 'COLAB_GPU' in os.environ:
  print("Executing in Colab!")
  # Cloning GitHub repository
  !git clone https://github.com/fwitschel/WIMA.git
  %cd WIMA


Executing in Colab!
Cloning into 'WIMA'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 163 (delta 55), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (163/163), 24.50 MiB | 12.58 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/WIMA/WIMA



First, we will install required dependencies.

In [27]:
!pip -q install openai qdrant-client pandas groq

In [28]:
import os
import pandas as pd

from tqdm.auto import tqdm

from openai import OpenAI

from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams,
    Distance,
    PointStruct
)

In [29]:
import pandas as pd
import csv
df = pd.read_table('/content/WIMA/data/publications-iwi.csv', sep="|")
df.head()

,Publikationstyp,Autorinnen und Autoren,Titel,Abstract,Übergeordnetes Werk,Verlag / herausgebende Institution,Publikationsjahr,Reviewform,Während FHNW Zugehörigkeit erstellt
0,05 - Bericht,"Leimstoll, Uwe; Wölfle, Ralf",Studien zum B2C-E-Commerce: Eine Übersicht übe...,NaN,NaN,NaN,2010,not peer-reviewed,unbekannt
1,04A - Beitrag Sammelband,"Bendel, Oliver",Die Industrie 4.0 aus Sicht der Ethik,NaN,"Industrie 4.0. Herausforderungen, Konzepte und...",Springer Viehweg,2017,peer-reviewed,Yes
2,05 - Bericht,"Hinkelmann, Knut; Thönssen, Barbara; Wolff, Da...",KISS - Knowledge-Intensive Service Support: An...,NaN,NaN,NaN,2007,not peer-reviewed,unbekannt
3,05 - Bericht,"Dornberger, Rolf",Cost of electricity analysis based on thermody...,NaN,NaN,NaN,2000,not peer-reviewed,unbekannt
4,04B - Beitrag Konferenzschrift,"Pimmer, Christoph; Gröhbiel, Urs; Walters, Fio...",The neglected grass root adoption of mobile ph...,Background/rationale: Many mhealth and mlearn...,NaN,NaN,2013,peer-reviewed,Yes


This function takes selected fields from each row and concatenates them into a text... This text is what will be searched - it doesn't need to contain all the metadata!
We will later - when creating the vector store - associate each vector with the full set of metadata so that all metadata can be retrieved.

In [30]:
MAX_ABSTRACT_CHARS = 500
def build_document(row):

    return f"""
Title:
{row.Titel}

Abstract:
{str(row["Abstract"])[:MAX_ABSTRACT_CHARS]}
""".strip()

We make a list of all the documents that shall be indexed in the vector store

In [31]:
documents = [
    build_document(row)
    for _, row in df.iterrows()
]

Connect to QDrant. You first need to:


*   log into Qdrant (e.g. with a Google account)
*   Go to "Clusters" and create a cluster
*   get the corresponding URL and API_KEY and store them in the "Secrets" here in Colab



In [32]:
from google.colab import userdata

QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_KEY")

qdrant = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

We create a collection in Qdrant (our vector store)

In [33]:
COLLECTION = "publications"

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(
        size=1024,
        distance=Distance.COSINE,
    ),
)

True

Next, we use JINA to map the documents (the textual part that we defined in the build_documents function) into the embedding space. To run this succesfully, log into JINA (e.g. using a Google account) and get yourself an API key. Place it into the "Secrets" on the left side (key icon) and use "JINA_KEY" as the name of the secret.

In [34]:
JINA_API_KEY = userdata.get("JINA_KEY")

client = OpenAI(
    api_key=JINA_API_KEY,
    base_url="https://api.jina.ai/v1"
)

Here, we define a function that will create the embeddings

In [35]:
MODEL = "jina-embeddings-v3"

def embed_documents(texts):

   response = client.embeddings.create(
    model=MODEL,
    input=texts,
    extra_body={
        "task": "retrieval.passage"
    }
  )

   return [item.embedding for item in response.data]

Here, we run the functions for embedding the text part of documents and then store the resulting vectors ("points") in a Qdrant vector store...

In [36]:
BATCH_SIZE = 64

for start in range(0, len(df), BATCH_SIZE):

    # Select the current batch
    batch = df.iloc[start:start + BATCH_SIZE]

    # Create the texts to be embedded
    texts = []
    for _, row in batch.iterrows():
        text = build_document(row)
        texts.append(text)

    # Generate embeddings
    embeddings = embed_documents(texts)

    # Create Qdrant points
    points = []

    for (_, row), embedding in zip(batch.iterrows(), embeddings):

        point = PointStruct(
            id=int(row.name),          # Unique document ID
            vector=embedding,           # Embedding vector
            payload=row.to_dict()       # Complete metadata
        )
        if row.name == 100:
          print(embedding)


        points.append(point)

    # Upload the batch to Qdrant
    qdrant.upsert(
        collection_name=COLLECTION,
        points=points
    )

print("Done!")

[0.03364786133170128, 0.05002497509121895, 0.01834007538855076, -0.024400770664215088, 0.037206340581178665, 0.09624181687831879, -0.04744404926896095, 0.000314911303576082, 0.04201142117381096, 0.01740485616028309, -0.09098196774721146, -0.005978307221084833, 0.04848378524184227, 0.08552178740501404, 0.011254651471972466, -0.034372977912425995, -0.046891625970602036, 0.04140239208936691, -0.14548340439796448, -0.0270133875310421, -0.10789314657449722, -0.06871677935123444, 0.004039127379655838, 0.04911202937364578, -0.021182497963309288, -0.05687306821346283, -0.017746014520525932, 0.01991472952067852, 0.05937426537275314, -0.11349882930517197, 0.03529766574501991, -0.009998620487749577, 0.002593601355329156, 0.06829209625720978, 0.07250246405601501, -0.016075434163212776, -0.07081953436136246, 0.04412112757563591, -0.15713928639888763, 0.07114149630069733, -0.04811418429017067, 0.042794447392225266, -0.08097366988658905, -0.02835163101553917, -0.08763690292835236, 0.00946864578872919

Queries also need to be projected into the embedding space:

In [37]:
def embed_query(query):

    response = client.embeddings.create(
        model=MODEL,
        input=[query],
        extra_body={
            "task": "retrieval.query"
        }
    )

    return response.data[0].embedding

We embed the query and then ask the vector store to return the top k results for it.

In [38]:
def search(query, topk):

    query_vector = embed_query(query)

    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=query_vector,
        limit=topk
    ).points

    return results

**Aufgabe**: Baut aus den Retrieval-Ergebnissen den Kontext zusammen, den das LLM bekommen soll, um die besten Experten zu einem Thema zu finden! -> Dazu an der markierten Stelle die Funktion bzw. den dort begonnen String erweitern!
Tip: die Variable "p" enhält alles, was ihr euch wünschen könnt. Mit p.get("X", "") könnt ihr das Metadatenfeld X auslesen.

In [39]:
def build_context(results):
    """
    Converts retrieved Qdrant results into a text that can be passed
    as context to an LLM.
    """

    context = []

    for i, hit in enumerate(results, start=1):

        p = hit.payload

        publication = f"""
        Publication {i}
        Similarity score: {hit.score:.3f}

        Title:
        {p.get("Titel", "")}

        Authors:
        {p.get("Autorinnen und Autoren", "")}

        Year:
        {p.get("Publikationsjahr", "")}
"""

        context.append(publication.strip())

    return "\n\n" + "=" * 80 + "\n\n".join(context)

Here, we can enter the topic we want to find experts for; it will then be searched for in the vector store...

In [40]:
topk = 42
query = " knowledge management"

context = search(query, topk=topk)
print(context)

[ScoredPoint(id=980, version=16, score=0.55641294, payload={'Publikationstyp': '04B - Beitrag Konferenzschrift', 'Autorinnen und Autoren': 'Hinkelmann, Knut', 'Titel': 'Wissensmanagement in der Praxis', 'Abstract': None, 'Übergeordnetes Werk': 'Kaderseminar der Bundeskanzlei, Sigriswil', 'Verlag / herausgebende Institution': None, 'Publikationsjahr': '2003', 'Reviewform': 'not peer-reviewed', 'Während FHNW Zugehörigkeit erstellt': 'unbekannt'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=944, version=15, score=0.54741824, payload={'Publikationstyp': '02 - Monographie', 'Autorinnen und Autoren': 'Hinkelmann, Knut; Wache, Holger', 'Titel': 'WM2009: 5th Conference on Professional Knowledge Management', 'Abstract': 'The Conference on Professional Knowledge Management provides a broad integrative overview of organizational, cultural, social and technical aspects on Knowledge Management. Focus of the conference is bringing together different research disciplines and sharin

After retrieving the relevant context, we build a prompt using this information and the user's original query. We will use a Llama model via the Groq API:



In [41]:
from groq import Groq
def llm(groq_client, prompt):
  chat_completion = groq_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="openai/gpt-oss-120b",
  )

  return chat_completion.choices[0].message.content

To make this run, log into Groq, get yourself an API key and store it as a secret. By now, you know how this works, right? :-)

In [42]:
groq_client = Groq(
    api_key=userdata.get('GROQ_API_KEY')
)

**Aufgabe**: gebt an der markierten Stelle den System Prompt an, mit dem das System angewiesen wird, unsere Expertensuche umzusetzen.

In [43]:
context_str = build_context(context)
prompt = f"""you are an assistant that helps users to find people with a certain expertise, based on their publications.
You need to analyse the publications and check who has most relevant, recent and impactful publications in the given field. Please combine all these factors into one judgement that you use to.
for each publication, the author is provided, along with publication year and venue: {context_str}. Please list up to 10 experts.
The topic that we are interested in is {query}"""
response = llm(groq_client, prompt)
print(response)

**Top‑10 Knowledge‑Management Experts (based on relevance, recency and impact of the supplied publication list)**  

| # | Expert | Why they rank high (combined relevance‑score, year and venue) |
|---|--------|----------------------------------------------------------------|
| 1 | **Knut Hinkelmann** | – Highest similarity scores (0.556 – 0.463) across many works.<br>– Continuous output from 2001 to **2026** (most recent: “Retaining explicit and implicit knowledge with RAG‑enhanced Generative AI”, 2026).<br>– Author of several well‑cited conference papers (WM2009, PROMOTE project) and journal/book chapters on process‑oriented KM, knowledge‑driven workflow, and KM‑enabled process flexibility. |
| 2 | **Sergej Miliaev** | – Co‑author of the 2026 RAG‑enhanced Generative‑AI KM paper (similarity 0.391).<br>– The work appears in a *high‑impact* venue (2026) that directly links modern AI techniques with knowledge‑management practice, signalling cutting‑edge impact. |
| 3 | **Andreas Martin** 